In [1]:
import pandas as pd
import numpy as np
 

df = pd.read_csv(r"C:\bank_loan_dataset.csv")
 
print("Initial shape:", df.shape)
print("\nMissing values before cleaning:\n", df.isna().sum())

Initial shape: (1000, 16)

Missing values before cleaning:
 Loan_ID               0
Gender               22
Married              35
Dependents           30
Education             0
Self_Employed        61
ApplicantIncome       0
CoapplicantIncome     0
LoanAmount            8
Loan_Amount_Term      0
Credit_History       83
Property_Area         0
Loan_Purpose          0
Interest_Rate         0
Application_Date      0
Loan_Status           0
dtype: int64


In [3]:
df["Gender"] = df["Gender"].replace({"M": "Male", "F": "Female"})
 
# Strip whitespace and standardize text case across object columns
categorical_cols = ["Gender", "Married", "Dependents", "Education",
                    "Self_Employed", "Property_Area", "Loan_Purpose", "Loan_Status"]
 
for col in categorical_cols:
    df[col] = df[col].astype(str).str.strip()
    df[col] = df[col].replace("nan", np.nan)  # restore true NaN after str conversion
 
# -------------------------------------------------
# 3. FIX INVALID / NEGATIVE VALUES
# -------------------------------------------------
# ApplicantIncome should never be negative -> take absolute value
df["ApplicantIncome"] = df["ApplicantIncome"].abs()
 
# -------------------------------------------------
# 4. HANDLE MISSING VALUES
# -------------------------------------------------
# Categorical columns -> fill with mode (most frequent value)
for col in ["Gender", "Married", "Self_Employed", "Dependents"]:
    df[col] = df[col].fillna(df[col].mode()[0])
 
# Credit_History -> fill with mode (it's effectively categorical: 0 or 1)
df["Credit_History"] = df["Credit_History"].fillna(df["Credit_History"].mode()[0])
 
# LoanAmount -> fill with median (robust to outliers, better than mean)
df["LoanAmount"] = df["LoanAmount"].fillna(df["LoanAmount"].median())
 
# -------------------------------------------------
# 5. FIX DATA TYPES
# -------------------------------------------------
# Auto-detect date format (handles DD-MM-YYYY or YYYY-MM-DD automatically)
try:
    df["Application_Date"] = pd.to_datetime(df["Application_Date"], format="%d-%m-%Y", errors="raise")
except (ValueError, TypeError):
    df["Application_Date"] = pd.to_datetime(df["Application_Date"], errors="coerce")
 
df["Credit_History"] = df["Credit_History"].astype(int)
df["Dependents"] = df["Dependents"].replace("3+", "3")  # for numeric analysis if needed
 
# -------------------------------------------------
# 6. REMOVE DUPLICATES
# -------------------------------------------------
before = len(df)
df = df.drop_duplicates(subset="Loan_ID")
print(f"\nRemoved {before - len(df)} duplicate rows")
 
# -------------------------------------------------
# 7. FEATURE ENGINEERING (useful for analysis/dashboards)
# -------------------------------------------------
df["Total_Income"] = df["ApplicantIncome"] + df["CoapplicantIncome"]
df["Loan_to_Income_Ratio"] = (df["LoanAmount"] * 1000) / df["Total_Income"]
 
# Recreate year/month AFTER the date column is correctly parsed
df["Application_Year"] = df["Application_Date"].dt.year
df["Application_Month"] = df["Application_Date"].dt.month_name()
 
df["Is_Default"] = df["Loan_Status"].map({"Charged Off": 1, "Fully Paid": 0, "Current": np.nan})
 
# -------------------------------------------------
# 8. FINAL CHECKS
# -------------------------------------------------
print("\nMissing values after cleaning:\n", df.isna().sum())
print("\nFinal shape:", df.shape)
print("\nData types:\n", df.dtypes)
 
# Sanity check: how many dates failed to parse, and why
failed_dates = df["Application_Date"].isna().sum()
print(f"\nRows with missing Application_Date: {failed_dates}")
print("(These should only be 'Current' status loans where date was blank in the original data)")
print(df.loc[df["Application_Date"].isna(), "Loan_Status"].value_counts())
 



Removed 0 duplicate rows

Missing values after cleaning:
 Loan_ID                  0
Gender                   0
Married                  0
Dependents               0
Education                0
Self_Employed            0
ApplicantIncome          0
CoapplicantIncome        0
LoanAmount               0
Loan_Amount_Term         0
Credit_History           0
Property_Area            0
Loan_Purpose             0
Interest_Rate            0
Application_Date         0
Loan_Status              0
Total_Income             0
Loan_to_Income_Ratio     0
Application_Year         0
Application_Month        0
Is_Default              94
dtype: int64

Final shape: (1000, 21)

Data types:
 Loan_ID                            str
Gender                             str
Married                            str
Dependents                         str
Education                          str
Self_Employed                      str
ApplicantIncome                  int64
CoapplicantIncome                int64
LoanAmount

In [4]:
df.to_csv(r"C:\excel project\bank_loan_dataset_cleaned.csv", index=False)
print("Cleaned file saved!")

Cleaned file saved!
